In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import sys, gc, glob, json, random, time
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segmentation-models-pytorch", "albumentations", "rasterio", 
                "scikit-learn", "timm"], capture_output=True)

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import rasterio
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings("ignore")

gc.collect()
torch.cuda.empty_cache()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n" + "="*80)
print("🚀 TRAINING ON GPU 1 - NOTEBOOK VERSION")
print("="*80)
print(f"\n📊 GPU Setup:")
print(f"   Device: {DEVICE}")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"   Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

BASE = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata"

CFG = {
    "img_dir": f"{BASE}/trainingdata/DATASET_1024/images",
    "mask_dir": f"{BASE}/trainingdata/DATASET_1024/masks",
    "save_dir": f"{BASE}/CHECKPOINTS_GPU1_NOTEBOOK",
    "num_classes": 9,
    "tile_size": 1024,
    "encoder": "mit_b3",
    "encoder_weights": "imagenet",
    "batch_size": 4,
    "accum_steps": 2,
    "num_epochs": 80,
    "lr": 5e-5,
    "weight_decay": 1e-2,
    "val_split": 0.2,
    "num_workers": 4,
    "ce_weight": 0.3,
    "dice_weight": 0.4,
    "focal_weight": 0.3,
    "patience": 20,
    "min_delta": 1e-4,
    "resume": True,
    "resume_ckpt": f"{BASE}/CHECKPOINTS_GPU1_NOTEBOOK/best_model.pth",
}

os.makedirs(CFG["save_dir"], exist_ok=True)

CLASS_NAMES = [
    "background", "built_up_area", "road", "road_centre_line",
    "water_body", "water_body_line", "waterbody_point",
    "utility", "utility_poly",
]

PIXEL_COUNTS = np.array([
    1_764_710_366, 693_864_908, 13_185_924, 199_538_875,
    83_284_429, 75_806_214, 543_668, 44_707_486, 1_650_674,
], dtype=np.float64)

ACTIVE_CLASSES = list(range(1, 9))

print(f"\n📋 Pixel Distribution:")
total = PIXEL_COUNTS.sum()
for i, (name, count) in enumerate(zip(CLASS_NAMES, PIXEL_COUNTS)):
    pct = 100 * count / total
    print(f"   {i} {name:22s}: {int(count):>14,}  ({pct:6.4f}%)")

raw_weights = 1.0 / np.log1p(PIXEL_COUNTS)
raw_weights /= raw_weights.mean()
raw_weights = np.clip(raw_weights, 0, 20.0)
class_weights = torch.tensor(raw_weights, dtype=torch.float32).to(DEVICE)

class SegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as src:
            img = src.read().astype(np.float32) / 255.0
            img = np.transpose(img, (1, 2, 0))

        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1).astype(np.int64)

        mask = np.clip(mask, 0, CFG["num_classes"] - 1)

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img = aug["image"]
            mask = aug["mask"].long()
        else:
            img = torch.from_numpy(img.transpose(2, 0, 1)).float()
            mask = torch.from_numpy(mask).long()

        return img, mask


def compute_sample_weights(mask_paths: list, save_dir: str) -> list:
    cache_path = os.path.join(save_dir, "sample_weights.json")
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            weights = json.load(f)
        if len(weights) == len(mask_paths):
            print(f"✅ Loaded cached sample weights")
            return weights

    print(f"🔢 Computing sample weights...")
    weights = []
    for p in tqdm(mask_paths, desc="   Weights", leave=False):
        with rasterio.open(p) as src:
            mask = src.read(1).flatten()
        rare = sum((mask == c).sum() for c in [6, 8, 2, 5])
        weights.append(float(rare) + 1.0)

    with open(cache_path, "w") as f:
        json.dump(weights, f)
    return weights


train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.OneOf([A.GridDistortion(num_steps=5, distort_limit=0.2),
             A.OpticalDistortion(distort_limit=0.15)], p=0.2),
    A.OneOf([A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
             A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
             A.CLAHE(clip_limit=4.0)], p=0.5),
    A.OneOf([A.GaussNoise(var_limit=(10, 50)),
             A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.4))], p=0.2),
    A.CoarseDropout(max_holes=6, max_height=64, max_width=64, fill_value=0, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print(f"\n📁 Loading data...")
all_imgs = sorted(glob.glob(f"{CFG['img_dir']}/*.tif"))
all_masks = sorted(glob.glob(f"{CFG['mask_dir']}/*.tif"))

print(f"   ✅ Found {len(all_imgs)} tiles")

indices = list(range(len(all_imgs)))
train_idx, val_idx = train_test_split(indices, test_size=CFG["val_split"], random_state=SEED)

train_imgs = [all_imgs[i] for i in train_idx]
train_masks = [all_masks[i] for i in train_idx]
val_imgs = [all_imgs[i] for i in val_idx]
val_masks = [all_masks[i] for i in val_idx]

train_dataset = SegDataset(train_imgs, train_masks, train_transform)
val_dataset = SegDataset(val_imgs, val_masks, val_transform)

sample_weights = compute_sample_weights(train_masks, CFG["save_dir"])
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_dataset), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], sampler=sampler,
                         num_workers=CFG["num_workers"], pin_memory=True, prefetch_factor=2,
                         persistent_workers=False, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=CFG["batch_size"], shuffle=False,
                       num_workers=CFG["num_workers"], pin_memory=True, prefetch_factor=2,
                       persistent_workers=True)

print(f"   Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

print(f"\n🧠 Loading model...")
model = smp.Segformer(encoder_name=CFG["encoder"], encoder_weights=CFG["encoder_weights"],
                      in_channels=3, classes=CFG["num_classes"], activation=None).to(DEVICE)

if hasattr(model.encoder, "set_grad_checkpointing"):
    model.encoder.set_grad_checkpointing(True)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   ✅ SegFormer-B3 loaded ({total_params:,} params)")

ce_loss = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
dice_loss = smp.losses.DiceLoss(mode="multiclass", classes=ACTIVE_CLASSES, smooth=1.0)
focal_loss = smp.losses.FocalLoss(mode="multiclass", gamma=3.0)

def criterion(outputs, masks):
    return (CFG["ce_weight"] * ce_loss(outputs, masks) +
            CFG["dice_weight"] * dice_loss(outputs, masks) +
            CFG["focal_weight"] * focal_loss(outputs, masks))

encoder_params, decoder_params = [], []
for name, param in model.named_parameters():
    (encoder_params if "encoder" in name else decoder_params).append(param)

optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": CFG["lr"], "weight_decay": CFG["weight_decay"]},
    {"params": decoder_params, "lr": CFG["lr"] * 5, "weight_decay": CFG["weight_decay"]},
])

def warmup_cosine(epoch, warmup=5, total=80):
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / (total - warmup)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,
    lr_lambda=lambda ep: warmup_cosine(ep, warmup=5, total=CFG["num_epochs"]))

scaler = GradScaler("cuda")

print(f"✅ Loss & Optimizer configured")

def compute_miou(preds, masks, active=ACTIVE_CLASSES):
    ious = {}
    for cls in active:
        pred_c = (preds == cls)
        true_c = (masks == cls)
        inter = (pred_c & true_c).sum().item()
        union = (pred_c | true_c).sum().item()
        if union > 0:
            ious[cls] = inter / union
    miou = float(np.mean(list(ious.values()))) if ious else 0.0
    return ious, miou

best_miou = 0.0
start_epoch = 0
history = {"train_loss": [], "val_loss": [], "val_miou": []}
patience_ct = 0

print("\n" + "="*80)
print(f"🚀 STARTING TRAINING")
print(f"   Epochs: 1 → {CFG['num_epochs']}")
print(f"   GPU: 1 (Idle V100)")
print(f"   Tiles: {len(all_imgs)}")
print("="*80 + "\n")

for epoch in range(start_epoch, CFG["num_epochs"]):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    epoch_start = time.time()

    model.train()
    train_loss = 0.0
    valid_steps = 0
    nan_batches = 0
    optimizer.zero_grad()

    for step, (imgs, masks) in enumerate(tqdm(train_loader, desc=f"Ep {epoch+1:03d} train", leave=False)):
        imgs = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        with autocast("cuda"):
            outputs = model(imgs)
            loss = criterion(outputs, masks) / CFG["accum_steps"]

        if torch.isnan(loss) or torch.isinf(loss):
            nan_batches += 1
            optimizer.zero_grad()
            torch.cuda.empty_cache()
            continue

        scaler.scale(loss).backward()

        if (step + 1) % CFG["accum_steps"] == 0:
            scaler.unscale_(optimizer)
            has_nan_grad = any(p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
                             for p in model.parameters())
            if has_nan_grad:
                nan_batches += 1
                optimizer.zero_grad()
                scaler.update()
                torch.cuda.empty_cache()
                continue

            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * CFG["accum_steps"]
        valid_steps += 1

    scheduler.step()

    if valid_steps == 0:
        print("💀 All NaN — stopping.")
        break

    train_loss /= valid_steps
    train_time = time.time() - epoch_start

    val_start = time.time()
    model.eval()
    val_loss = 0.0
    val_steps = 0
    all_ious = {c: [] for c in ACTIVE_CLASSES}

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Ep {epoch+1:03d} val  ", leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            with autocast("cuda"):
                outputs = model(imgs)
                loss = criterion(outputs, masks)

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            preds = torch.argmax(outputs, dim=1)
            val_loss += loss.item()
            val_steps += 1
            ious, _ = compute_miou(preds, masks)
            for cls, iou in ious.items():
                all_ious[cls].append(iou)

    val_time = time.time() - val_start
    val_loss /= max(val_steps, 1)
    per_class_iou = {c: float(np.mean(v)) for c, v in all_ious.items() if v}
    val_miou = float(np.mean(list(per_class_iou.values()))) if per_class_iou else 0.0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9

    torch.cuda.empty_cache()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(val_miou)

    improved = val_miou > best_miou + CFG["min_delta"]
    if improved:
        best_miou = val_miou
        patience_ct = 0
        torch.save({
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "val_miou": val_miou,
            "cfg": CFG,
            "active_classes": ACTIVE_CLASSES,
            "class_names": CLASS_NAMES,
            "pixel_counts": PIXEL_COUNTS.tolist(),
        }, os.path.join(CFG["save_dir"], "best_model.pth"))
        tag = f"  ✅ best={best_miou:.4f}"
    else:
        patience_ct += 1
        tag = f"  (pat {patience_ct}/{CFG['patience']})"

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Ep {epoch+1:3d}/{CFG['num_epochs']} | Train: {train_loss:.4f} ({train_time:5.0f}s) | "
          f"Val: {val_loss:.4f} ({val_time:5.0f}s) | mIoU: {val_miou:.4f} | VRAM: {peak_mem:5.1f}GB | "
          f"LR: {lr_now:.2e}{tag}")

    if (epoch + 1) % 10 == 0:
        print("   Per-class IoU:")
        for cls in ACTIVE_CLASSES:
            iou = per_class_iou.get(cls, 0.0)
            print(f"      {cls} {CLASS_NAMES[cls]:22s}: {iou:.4f}")

    if patience_ct >= CFG["patience"]:
        print(f"\n⏹️  Early stopping at epoch {epoch + 1}.")
        break

print(f"\n{'='*80}")
print(f"✅ Training completed!")
print(f"   Best mIoU: {best_miou:.4f}")
print(f"   Saved to: {CFG['save_dir']}")
print(f"{'='*80}\n")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history["train_loss"]) + 1)

ax1.plot(ep, history["train_loss"], label="Train", linewidth=2)
ax1.plot(ep, history["val_loss"], label="Val", linewidth=2)
ax1.set_title("Loss", fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(ep, history["val_miou"], color="green", label="Val mIoU", linewidth=2)
ax2.set_title("Validation mIoU", fontweight='bold')
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CFG["save_dir"], "training_curves.png"), dpi=150)
print("📊 Training curves saved!")
plt.show()

print(f"✅ Done! Check {CFG['save_dir']} for results")


🚀 TRAINING ON GPU 1 - NOTEBOOK VERSION

📊 GPU Setup:
   Device: cuda
   GPU: Tesla V100-PCIE-32GB
   Memory: 34.1 GB
   Free: 33.7 GB

📋 Pixel Distribution:
   0 background            :  1,764,710,366  (61.3323%)
   1 built_up_area         :    693,864,908  (24.1152%)
   2 road                  :     13,185,924  (0.4583%)
   3 road_centre_line      :    199,538,875  (6.9350%)
   4 water_body            :     83,284,429  (2.8945%)
   5 water_body_line       :     75,806,214  (2.6346%)
   6 waterbody_point       :        543,668  (0.0189%)
   7 utility               :     44,707,486  (1.5538%)
   8 utility_poly          :      1,650,674  (0.0574%)

📁 Loading data...
   ✅ Found 2744 tiles
✅ Loaded cached sample weights
   Train batches: 548, Val batches: 138

🧠 Loading model...
   ✅ SegFormer-B3 loaded (44,600,265 params)
✅ Loss & Optimizer configured

🚀 STARTING TRAINING
   Epochs: 1 → 80
   GPU: 1 (Idle V100)
   Tiles: 2744



Ep 001 train:   0%|          | 0/548 [00:00<?, ?it/s]

Ep 001 val  :   0%|          | 0/138 [00:00<?, ?it/s]

Ep   1/80 | Train: 1.0263 (  357s) | Val: 0.8989 (   35s) | mIoU: 0.0420 | VRAM:  30.7GB | LR: 2.00e-05  ✅ best=0.0420


Ep 002 train:   0%|          | 0/548 [00:00<?, ?it/s]

...

Ep 003 train:   0%|          | 0/548 [00:00<?, ?it/s]

Ep 003 val  :   0%|          | 0/138 [00:00<?, ?it/s]

Ep   3/80 | Train: 0.7360 (  315s) | Val: 0.6096 (   28s) | mIoU: 0.1121 | VRAM:  30.5GB | LR: 4.00e-05  ✅ best=0.1121


Ep 004 train:   0%|          | 0/548 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
"""
4-CLASS SEGMENTATION TRAINING
==============================
Classes: background, built_up_area, road_centre_line, water_body, water_body_line
Metric : Pixel Accuracy (total model accuracy)
Expected: 88-94% pixel accuracy

All rare classes remapped to background — model focuses only on
the 4 classes with enough data to learn properly.
"""

import os, gc, glob, json, random, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segmentation-models-pytorch", "albumentations",
                "rasterio", "scikit-learn", "timm"])

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

gc.collect()
torch.cuda.empty_cache()

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark    = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32   = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Free   : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

BASE = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata"

# ═══════════════════════════════════════════════════════════════════════════════
# CLASS REMAPPING
# Original 9 classes → 5 classes (background + 4 top classes)
#
# Original → New
#   0 background       → 0 background
#   1 built_up_area    → 1 built_up_area    (24.12% pixels)
#   2 road             → 0 background       (too sparse — 0.46%)
#   3 road_centre_line → 2 road_centre_line (6.94% pixels)
#   4 water_body       → 3 water_body       (2.89% pixels)
#   5 water_body_line  → 4 water_body_line  (2.63% pixels)
#   6 waterbody_point  → 0 background       (too sparse)
#   7 utility          → 0 background       (too sparse)
#   8 utility_poly     → 0 background       (too sparse)
# ═══════════════════════════════════════════════════════════════════════════════

# Remap table: original_class_id → new_class_id
REMAP = {
    0: 0,   # background     → background
    1: 1,   # built_up_area  → built_up_area
    2: 0,   # road           → background (sparse)
    3: 2,   # road_centre_line → road_centre_line
    4: 3,   # water_body     → water_body
    5: 4,   # water_body_line → water_body_line
    6: 0,   # waterbody_point → background (sparse)
    7: 0,   # utility        → background (sparse)
    8: 0,   # utility_poly   → background (sparse)
}

# Build fast numpy lookup array
REMAP_ARRAY = np.zeros(9, dtype=np.int64)
for orig, new in REMAP.items():
    REMAP_ARRAY[orig] = new

NUM_CLASSES  = 5
CLASS_NAMES  = [
    "background",
    "built_up_area",
    "road_centre_line",
    "water_body",
    "water_body_line",
]

COLOR_MAP = np.array([
    [0,   0,   0  ],   # 0 background
    [255, 0,   0  ],   # 1 built_up_area
    [255, 165, 0  ],   # 2 road_centre_line
    [0,   0,   255],   # 3 water_body
    [0,   150, 255],   # 4 water_body_line
], dtype=np.uint8)

# Pixel counts AFTER remapping (background absorbs sparse classes)
PIXEL_COUNTS = np.array([
    1_764_710_366 + 13_185_924 + 543_668 + 44_707_486 + 1_650_674,  # 0 bg
    693_864_908,   # 1 built_up_area
    199_538_875,   # 2 road_centre_line
     83_284_429,   # 3 water_body
     75_806_214,   # 4 water_body_line
], dtype=np.float64)

ACTIVE_CLASSES = [1, 2, 3, 4]   # exclude background from mIoU

print("\n4-Class setup after remapping:")
total = PIXEL_COUNTS.sum()
for i, (n, c) in enumerate(zip(CLASS_NAMES, PIXEL_COUNTS)):
    print(f"  {i} {n:22s}: {int(c):>14,}  ({100*c/total:.3f}%)")

# ── CLASS WEIGHTS ─────────────────────────────────────────────────────────────
raw_weights   = 1.0 / np.log1p(PIXEL_COUNTS)
raw_weights  /= raw_weights.mean()
raw_weights   = np.clip(raw_weights, 0, 10.0)
class_weights = torch.tensor(raw_weights, dtype=torch.float32).to(DEVICE)

print("\nClass weights:")
for i, (n, w) in enumerate(zip(CLASS_NAMES, raw_weights)):
    print(f"  {i} {n:22s}: {w:.4f}")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
CFG = {
    "img_dir":          f"{BASE}/trainingdata/DATASET_1024/images",
    "mask_dir":         f"{BASE}/trainingdata/DATASET_1024/masks",
    "save_dir":         f"{BASE}/CHECKPOINTS_4CLASS",

    "num_classes":      NUM_CLASSES,
    "tile_size":        1024,

    "encoder":          "mit_b3",
    "encoder_weights":  "imagenet",

    "batch_size":       4,
    "accum_steps":      2,          # effective batch = 8
    "num_epochs":       80,
    "lr":               5e-5,
    "weight_decay":     1e-2,
    "val_split":        0.2,
    "num_workers":      4,

    "ce_weight":        0.4,
    "dice_weight":      0.4,
    "focal_weight":     0.2,        # lower focal — less class imbalance now

    "patience":         20,
    "min_delta":        1e-4,

    "resume":           False,
    "resume_ckpt":      f"{BASE}/CHECKPOINTS_4CLASS/best_model.pth",
}

os.makedirs(CFG["save_dir"], exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════════════════
class SegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.transform  = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as src:
            img = src.read().astype(np.float32) / 255.0
            img = np.transpose(img, (1, 2, 0))          # H W C

        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1).astype(np.int64)

        # Remap 9 original classes → 5 new classes
        mask = np.clip(mask, 0, 8)
        mask = REMAP_ARRAY[mask]

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug["image"]
            mask = aug["mask"].long()
        else:
            img  = torch.from_numpy(img.transpose(2, 0, 1)).float()
            mask = torch.from_numpy(mask).long()

        return img, mask

    def get_sample_weight(self, idx):
        """Oversample tiles with active classes."""
        with rasterio.open(self.mask_paths[idx]) as src:
            mask = REMAP_ARRAY[np.clip(src.read(1).flatten(), 0, 8)]
        fg = (mask > 0).sum()
        return float(fg) + 1.0

# ── AUGMENTATIONS ─────────────────────────────────────────────────────────────
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                       rotate_limit=15, p=0.4),
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.2),
        A.OpticalDistortion(distort_limit=0.15),
    ], p=0.2),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        A.CLAHE(clip_limit=4.0),
    ], p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ── DATALOADERS ───────────────────────────────────────────────────────────────
all_imgs  = sorted(glob.glob(f"{CFG['img_dir']}/*.tif"))
all_masks = sorted(glob.glob(f"{CFG['mask_dir']}/*.tif"))

assert len(all_imgs) > 0, f"No tiles in {CFG['img_dir']}"
assert len(all_imgs) == len(all_masks)

print(f"\nTotal tiles : {len(all_imgs)}")

indices = list(range(len(all_imgs)))
train_idx, val_idx = train_test_split(
    indices, test_size=CFG["val_split"], random_state=SEED
)

train_imgs  = [all_imgs[i]  for i in train_idx]
train_masks = [all_masks[i] for i in train_idx]
val_imgs    = [all_imgs[i]  for i in val_idx]
val_masks   = [all_masks[i] for i in val_idx]

for name, data in [("train_imgs", train_imgs), ("train_masks", train_masks),
                   ("val_imgs",   val_imgs),   ("val_masks",   val_masks)]:
    with open(os.path.join(CFG["save_dir"], f"{name}.json"), "w") as f:
        json.dump(data, f)

train_dataset = SegDataset(train_imgs, train_masks, train_transform)
val_dataset   = SegDataset(val_imgs,   val_masks,   val_transform)

# Cache sample weights
cache_path = os.path.join(CFG["save_dir"], "sample_weights.json")
if os.path.exists(cache_path):
    with open(cache_path) as f:
        sample_weights = json.load(f)
    if len(sample_weights) != len(train_dataset):
        sample_weights = None
    else:
        print("Loaded cached sample weights")
else:
    sample_weights = None

if sample_weights is None:
    print("Computing sample weights...")
    sample_weights = [
        train_dataset.get_sample_weight(i)
        for i in tqdm(range(len(train_dataset)), desc="Weights")
    ]
    with open(cache_path, "w") as f:
        json.dump(sample_weights, f)

sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(train_dataset), replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=CFG["batch_size"], sampler=sampler,
    num_workers=CFG["num_workers"], pin_memory=True,
    prefetch_factor=2, persistent_workers=False, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG["batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
    prefetch_factor=2, persistent_workers=True,
)

print(f"Train tiles : {len(train_dataset)}")
print(f"Val tiles   : {len(val_dataset)}")

# ═══════════════════════════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════════════════════════
model = smp.Segformer(
    encoder_name    = CFG["encoder"],
    encoder_weights = CFG["encoder_weights"],
    in_channels     = 3,
    classes         = CFG["num_classes"],
    activation      = None,
).to(DEVICE)

if hasattr(model.encoder, "set_grad_checkpointing"):
    model.encoder.set_grad_checkpointing(True)
    print("Gradient checkpointing enabled")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model      : SegFormer-B3 ({total_params:,} params)")
print(f"GPU free   : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ═══════════════════════════════════════════════════════════════════════════════
# LOSS
# ═══════════════════════════════════════════════════════════════════════════════
ce_loss    = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
dice_loss  = smp.losses.DiceLoss(
    mode="multiclass", classes=ACTIVE_CLASSES, smooth=1.0
)
focal_loss = smp.losses.FocalLoss(mode="multiclass", gamma=2.0)

def criterion(outputs, masks):
    return (CFG["ce_weight"]    * ce_loss(outputs, masks)    +
            CFG["dice_weight"]  * dice_loss(outputs, masks)  +
            CFG["focal_weight"] * focal_loss(outputs, masks))

# ═══════════════════════════════════════════════════════════════════════════════
# OPTIMIZER + SCHEDULER
# ═══════════════════════════════════════════════════════════════════════════════
encoder_params, decoder_params = [], []
for name, param in model.named_parameters():
    (encoder_params if "encoder" in name else decoder_params).append(param)

optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": CFG["lr"],     "weight_decay": CFG["weight_decay"]},
    {"params": decoder_params, "lr": CFG["lr"] * 5, "weight_decay": CFG["weight_decay"]},
])

def warmup_cosine(epoch, warmup=5, total=80):
    if epoch < warmup:
        return (epoch + 1) / warmup
    return 0.5 * (1.0 + np.cos(np.pi * (epoch - warmup) / (total - warmup)))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda ep: warmup_cosine(ep, warmup=5, total=CFG["num_epochs"]),
)

scaler = GradScaler("cuda")

# ═══════════════════════════════════════════════════════════════════════════════
# RESUME
# ═══════════════════════════════════════════════════════════════════════════════
best_miou   = 0.0
best_acc    = 0.0
start_epoch = 0

if CFG["resume"] and os.path.exists(CFG["resume_ckpt"]):
    ckpt  = torch.load(CFG["resume_ckpt"], map_location=DEVICE, weights_only=False)
    state = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model_state"].items()}
    model.load_state_dict(state, strict=False)
    best_miou   = ckpt.get("val_miou", 0.0)
    best_acc    = ckpt.get("val_acc",  0.0)
    start_epoch = ckpt["epoch"]
    print(f"\n✅ Resumed epoch {start_epoch} | mIoU={best_miou:.4f} | Acc={best_acc:.4f}")
else:
    print("\nTraining from scratch")

# ═══════════════════════════════════════════════════════════════════════════════
# METRICS — pixel accuracy + mIoU
# ═══════════════════════════════════════════════════════════════════════════════
def compute_metrics(preds, masks):
    """Returns pixel accuracy and per-class IoU."""
    # Pixel accuracy — fraction of correctly classified pixels
    correct = (preds == masks).sum().item()
    total   = masks.numel()
    acc     = correct / total

    # Per-class IoU
    ious = {}
    for cls in ACTIVE_CLASSES:
        pred_c = (preds == cls)
        true_c = (masks == cls)
        inter  = (pred_c & true_c).sum().item()
        union  = (pred_c | true_c).sum().item()
        if union > 0:
            ious[cls] = inter / union

    miou = float(np.mean(list(ious.values()))) if ious else 0.0
    return acc, ious, miou

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════
history           = {"train_loss": [], "val_loss": [],
                     "val_miou":   [], "val_acc":  []}
patience_ct       = 0
nan_batches_total = 0

print("\n" + "═" * 70)
print(f"4-Class model | {len(all_imgs)} tiles | {CFG['tile_size']}px")
print(f"Classes : {CLASS_NAMES}")
print(f"Target  : Pixel Accuracy > 90%")
print(f"Epochs  : {start_epoch+1} → {CFG['num_epochs']}")
print("═" * 70)

for epoch in range(start_epoch, CFG["num_epochs"]):

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss  = 0.0
    valid_steps = 0
    nan_batches = 0
    optimizer.zero_grad()

    for step, (imgs, masks) in enumerate(tqdm(
            train_loader, desc=f"Ep {epoch+1:03d} train", leave=False)):

        imgs  = imgs.to(DEVICE,  non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        with autocast("cuda"):
            outputs = model(imgs)
            loss    = criterion(outputs, masks) / CFG["accum_steps"]

        if torch.isnan(loss) or torch.isinf(loss):
            nan_batches += 1
            optimizer.zero_grad()
            torch.cuda.empty_cache()
            continue

        scaler.scale(loss).backward()

        if (step + 1) % CFG["accum_steps"] == 0:
            scaler.unscale_(optimizer)
            has_nan = any(
                p.grad is not None and
                (torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
                for p in model.parameters()
            )
            if has_nan:
                nan_batches += 1
                optimizer.zero_grad()
                scaler.update()
                torch.cuda.empty_cache()
                continue

            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss  += loss.item() * CFG["accum_steps"]
        valid_steps += 1

    scheduler.step()

    if nan_batches > 0:
        nan_batches_total += nan_batches
        print(f"  ⚠️  {nan_batches} NaN batches (total: {nan_batches_total})")
    if valid_steps == 0:
        print("  💀 All NaN — stopping."); break

    train_loss /= valid_steps
    train_time  = time.time() - t0

    # ── VALIDATE ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss  = 0.0
    val_steps = 0
    all_ious  = {c: [] for c in ACTIVE_CLASSES}
    all_accs  = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader,
                                desc=f"Ep {epoch+1:03d} val  ", leave=False):
            imgs  = imgs.to(DEVICE,  non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            with autocast("cuda"):
                outputs = model(imgs)
                loss    = criterion(outputs, masks)

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            preds      = torch.argmax(outputs, dim=1)
            val_loss  += loss.item()
            val_steps += 1

            acc, ious, _ = compute_metrics(preds, masks)
            all_accs.append(acc)
            for cls, iou in ious.items():
                all_ious[cls].append(iou)

    val_time      = time.time() - t0 - train_time
    val_loss     /= max(val_steps, 1)
    per_class_iou = {c: float(np.mean(v)) for c, v in all_ious.items() if v}
    val_miou      = float(np.mean(list(per_class_iou.values()))) if per_class_iou else 0.0
    val_acc       = float(np.mean(all_accs)) if all_accs else 0.0
    peak_mem      = torch.cuda.max_memory_allocated() / 1e9

    torch.cuda.empty_cache()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(val_miou)
    history["val_acc"].append(val_acc)

    # Save on best accuracy (your metric) OR best mIoU
    improved = val_acc > best_acc + CFG["min_delta"]
    if improved:
        best_acc    = val_acc
        best_miou   = val_miou
        patience_ct = 0
        torch.save({
            "epoch":          epoch + 1,
            "model_state":    model.state_dict(),
            "optim_state":    optimizer.state_dict(),
            "val_miou":       val_miou,
            "val_acc":        val_acc,
            "cfg":            CFG,
            "active_classes": ACTIVE_CLASSES,
            "class_names":    CLASS_NAMES,
            "remap":          REMAP,
        }, os.path.join(CFG["save_dir"], "best_model.pth"))
        tag = f"  ✅ best acc={best_acc:.4f} mIoU={best_miou:.4f}"
    else:
        patience_ct += 1
        tag = f"  (patience {patience_ct}/{CFG['patience']})"

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Ep {epoch+1:03d}/{CFG['num_epochs']} | "
          f"Loss: {train_loss:.4f} | Val: {val_loss:.4f} | "
          f"Acc: {val_acc*100:.2f}% | mIoU: {val_miou:.4f} | "
          f"VRAM: {peak_mem:.1f}GB | LR: {lr_now:.2e}{tag}")

    if (epoch + 1) % 10 == 0:
        print("  Per-class IoU:")
        for cls in ACTIVE_CLASSES:
            iou = per_class_iou.get(cls, 0.0)
            print(f"    {cls} {CLASS_NAMES[cls]:22s}: {iou:.4f}")

    if patience_ct >= CFG["patience"]:
        print(f"\nEarly stopping at epoch {epoch+1}.")
        break

print(f"\nBest Pixel Accuracy : {best_acc*100:.2f}%")
print(f"Best Val mIoU       : {best_miou:.4f}")

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING CURVES
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(history["train_loss"]) + 1)

axes[0].plot(ep, history["train_loss"], label="Train")
axes[0].plot(ep, history["val_loss"],   label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(ep, [v*100 for v in history["val_acc"]], color="blue", label="Pixel Acc %")
axes[1].axhline(90, color="red", linestyle="--", label="90% target")
axes[1].axhline(95, color="green", linestyle="--", label="95% target")
axes[1].set_title("Pixel Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")

axes[2].plot(ep, history["val_miou"], color="green", label="mIoU")
axes[2].set_title("Val mIoU"); axes[2].legend(); axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.savefig(os.path.join(CFG["save_dir"], "training_curves.png"), dpi=150)
plt.show()
print(f"Saved → {CFG['save_dir']}/training_curves.png")

Device : cuda
GPU    : Tesla V100-PCIE-32GB
Memory : 34.1 GB
Free   : 32.1 GB

4-Class setup after remapping:
  0 background            :  1,824,798,118  (63.421%)
  1 built_up_area         :    693,864,908  (24.115%)
  2 road_centre_line      :    199,538,875  (6.935%)
  3 water_body            :     83,284,429  (2.895%)
  4 water_body_line       :     75,806,214  (2.635%)

Class weights:
  0 background            : 0.9078
  1 built_up_area         : 0.9509
  2 road_centre_line      : 1.0129
  3 water_body            : 1.0614
  4 water_body_line       : 1.0669

Total tiles : 2744
Computing sample weights...


Weights: 100%|██████████| 2195/2195 [00:17<00:00, 122.02it/s]


Train tiles : 2195
Val tiles   : 549
Model      : SegFormer-B3 (44,599,237 params)
GPU free   : 32.0 GB

Training from scratch

══════════════════════════════════════════════════════════════════════
4-Class model | 2744 tiles | 1024px
Classes : ['background', 'built_up_area', 'road_centre_line', 'water_body', 'water_body_line']
Target  : Pixel Accuracy > 90%
Epochs  : 1 → 80
══════════════════════════════════════════════════════════════════════


  ⚠️  2 NaN batches (total: 2)


Ep 001/80 | Loss: 0.9336 | Val: 0.7587 | Acc: 64.28% | mIoU: 0.1120 | VRAM: 30.4GB | LR: 2.00e-05  ✅ best acc=0.6428 mIoU=0.1120


...

Ep 026/80 | Loss: 0.2791 | Val: 0.3127 | Acc: 91.36% | mIoU: 0.6103 | VRAM: 30.4GB | LR: 4.09e-05  ✅ best acc=0.9136 mIoU=0.6103


Ep 027/80 | Loss: 0.2804 | Val: 0.3184 | Acc: 91.23% | mIoU: 0.5888 | VRAM: 30.4GB | LR: 4.01e-05  (patience 1/20)


Ep 028/80 | Loss: 0.2745 | Val: 0.3068 | Acc: 91.43% | mIoU: 0.5940 | VRAM: 30.4GB | LR: 3.93e-05  ✅ best acc=0.9143 mIoU=0.5940


Ep 029/80 | Loss: 0.2669 | Val: 0.3435 | Acc: 89.93% | mIoU: 0.5605 | VRAM: 30.4GB | LR: 3.84e-05  (patience 1/20)


Ep 030 train:  23%|██▎       | 124/548 [01:11<03:50,  1.84it/s]

In [ ]:
import subprocess, os

# Show current notebook PID
print(f"THIS notebook PID: {os.getpid()}")
